# Pipeline Step 03 — Google News BERT Sentiment Scoring

**Run on Google Colab with GPU runtime (T4 is sufficient).**

## What this notebook does
Scores each Google News article title with a Turkish BERT sentiment model and computes yearly aggregate negativity scores. These scores feed the `d1_bert_mean` and `d1_bert_weighted` D1 alternatives (not the primary D1 — see CONTEXT.md).

## Inputs
- `gnews_scored.csv` — article-level data from step 02, uploaded to Google Drive  
  Required column: `title_clean` (lowercased, punctuation-stripped article titles)

## Outputs
- `gnews_bert_scored.csv` — title-level BERT negativity probabilities  
- `gnews_yearly_bert_sentiment.csv` — yearly aggregates (`bert_mean_neg`, `d1_bert_weighted`)

## Setup
1. Upload `gnews_scored.csv` to your Google Drive folder
2. Set `DRIVE_FOLDER` below to match your Drive path
3. Runtime → Change runtime type → GPU (T4)
4. Run all cells

## Model
`savasy/bert-base-turkish-sentiment-cased` — label 0 = negative, label 1 = positive  
(Confirmed via label direction check cell below)

In [ ]:
!pip install transformers torch pandas tqdm -q

In [ ]:
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
from google.colab import drive

# ── Mount Drive ────────────────────────────────────────────────────────────
drive.mount("/content/drive")

# ── PATH CONFIG — set DRIVE_FOLDER to your own Google Drive path ───────────
DRIVE_FOLDER  = "/content/drive/MyDrive/nlp_project"
INPUT_PATH    = f"{DRIVE_FOLDER}/gnews_scored.csv"
OUTPUT_SCORED = f"{DRIVE_FOLDER}/gnews_bert_scored.csv"
OUTPUT_YEARLY = f"{DRIVE_FOLDER}/gnews_yearly_bert_sentiment.csv"
# ──────────────────────────────────────────────────────────────────────────

MODEL_NAME  = "savasy/bert-base-turkish-sentiment-cased"
BATCH_SIZE  = 128
NEG_LABEL_IDX = 0   # label 0 = negative (confirmed by label direction check below)

In [ ]:
# Load data — keep only treatment group articles
df = pd.read_csv(INPUT_PATH)
treatment = df[df["group"] == "treatment"].copy()
print(f"Total rows    : {len(df)}")
print(f"Treatment rows: {len(treatment)}")

In [ ]:
# Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

In [ ]:
# Label direction check — run before scoring to confirm NEG_LABEL_IDX = 0
# Expected: label_0 HIGH for violent sentence, LOW for positive sentence
test_cases = [
    "doktora şiddet uygulandı, saldırgan tutuklandı",
    "başarılı ameliyat, hasta taburcu edildi",
]
print("=== Label direction check ===")
for sent in test_cases:
    enc = tokenizer(sent, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model(**enc)
    probs = F.softmax(out.logits, dim=-1).cpu()
    print(f"  '{sent}'")
    print(f"  label_0: {probs[0][0]:.3f}  label_1: {probs[0][1]:.3f}")
print("\nIf label_0 is NOT high for the violent sentence, set NEG_LABEL_IDX = 1 above.")

In [ ]:
# Inference
texts     = treatment["title_clean"].fillna("").tolist()
neg_probs = []

with torch.no_grad():
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Scoring"):
        batch   = texts[i : i + BATCH_SIZE]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors="pt",
        ).to(device)
        outputs = model(**encoded)
        probs   = F.softmax(outputs.logits, dim=-1)
        neg_probs.extend(probs[:, NEG_LABEL_IDX].cpu().numpy().tolist())

treatment["bert_neg_prob"] = neg_probs

In [ ]:
# Yearly aggregation
yearly = (
    treatment.groupby("year")
    .agg(
        article_count      = ("title_clean",   "count"),
        bert_mean_neg      = ("bert_neg_prob",  "mean"),
        bert_neg_flag_rate = ("bert_neg_prob",  lambda x: (x > 0.5).mean()),
    )
    .reset_index()
)
yearly["d1_bert_weighted"] = yearly["article_count"] * yearly["bert_mean_neg"]

print("=== BERT yearly scores ===")
print(yearly.to_string(index=False))

In [ ]:
# Save outputs
treatment.to_csv(OUTPUT_SCORED, index=False)
yearly.to_csv(OUTPUT_YEARLY, index=False)
print(f"Saved: {OUTPUT_SCORED}")
print(f"Saved: {OUTPUT_YEARLY}")